In [5]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import SimpleRNN, Dense

# Dataset configuration
SUNSPOTS_URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/monthly-sunspots.csv'
TIME_STEPS = 12  # Using 12 months for annual pattern recognition
TRAIN_SPLIT = 0.8  # 80% training, 20% testing

def load_and_preprocess_data():
    """Load and normalize sunspot dataset"""
    df = pd.read_csv(SUNSPOTS_URL, usecols=['Sunspots'])
    data = df.values.astype('float32')
    
    # Normalization to [0,1] range
    scaler = MinMaxScaler(feature_range=(0, 1))
    scaled_data = scaler.fit_transform(data).flatten()
    
    return scaled_data, scaler

def create_sequences(data, time_steps):
    """Convert time series into supervised learning format"""
    X, y = [], []
    for i in range(len(data) - time_steps):
        X.append(data[i:(i + time_steps)])
        y.append(data[i + time_steps])
    return np.array(X), np.array(y)

def build_simple_rnn_model(input_shape):
    """Construct SimpleRNN model architecture"""
    model = Sequential([
        SimpleRNN(64, activation='tanh', 
                input_shape=input_shape,
                kernel_initializer='glorot_uniform',
                recurrent_initializer='orthogonal'),
        Dense(1, activation='linear')
    ])
    
    model.compile(optimizer='adam', loss='mean_squared_error')
    return model

def evaluate_model(model, X_test, y_test, scaler):
    """Calculate and display performance metrics"""
    predictions = model.predict(X_test)
    
    # Inverse transform to original scale
    y_test_inv = scaler.inverse_transform(y_test.reshape(-1, 1))
    pred_inv = scaler.inverse_transform(predictions)
    
    rmse = math.sqrt(mean_squared_error(y_test_inv, pred_inv))
    print(f'\nTest RMSE: {rmse:.3f}')
    return pred_inv

def plot_results(y_true, predictions, time_steps):
    """Visualize predictions vs actual values"""
    plt.figure(figsize=(14, 6))
    plt.plot(y_true, label='Actual Sunspots', alpha=0.7)
    plt.plot(predictions, label='RNN Predictions', linestyle='--')
    plt.title(f'SimpleRNN Forecasting ({time_steps}-Month Lookback)')
    plt.xlabel('Month Sequence')
    plt.ylabel('Sunspot Count')
    plt.legend()
    plt.grid(True)
    plt.show()

# Main execution pipeline
if __name__ == "__main__":
    # Data preparation
    data, scaler = load_and_preprocess_data()
    X, y = create_sequences(data, TIME_STEPS)
    
    # Reshape for RNN input: [samples, time_steps, features]
    X = X.reshape((X.shape[0], X.shape[1], 1))
    
    # Train-test split
    split_idx = int(len(X) * TRAIN_SPLIT)
    X_train, X_test = X[:split_idx], X[split_idx:]
    y_train, y_test = y[:split_idx], y[split_idx:]
    
    # Model construction
    model = build_simple_rnn_model((TIME_STEPS, 1))
    print(model.summary())
    
    # Model training
    history = model.fit(X_train, y_train,
                       epochs=100,
                       batch_size=32,
                       validation_split=0.2,
                       verbose=1)
    
    # Evaluation and visualization
    predictions = evaluate_model(model, X_test, y_test, scaler)
    plot_results(scaler.inverse_transform(y_test.reshape(-1,1)), predictions, TIME_STEPS)
